# Máy AI cá nhân StoryForge

Đây là mẫu công khai từ GitHub, có sẵn mã và biểu mẫu để chạy. Mở mẫu không tự tạo file trong Drive.
Bạn có thể chọn **Tệp → Lưu bản sao vào Drive** để giữ chỉnh sửa; bước này **không bắt buộc** mỗi ngày.
Chỉnh sửa trong phiên hoặc bản sao của bạn không sửa nguồn GitHub nếu bạn không có quyền ghi vào kho.
Mỗi người chạy runtime và liên kết riêng, không dùng chung GPU, key hoặc tunnel của người khác.
Phiên runtime mới không mang theo model đã tải, RAM, tiến trình hay credential của phiên cũ.

## Bắt đầu

1. Chọn runtime GPU hoặc CPU, mở notebook rồi chạy ô điều khiển.
2. Chọn **Ghép nối / khởi động**. Notebook tạo mã dùng một lần và chờ tối đa 10 phút.
3. Trong StoryForge, mở **Cài đặt → Máy AI cá nhân**, nhập mã và xác nhận kết nối.
4. Khi trạng thái web và notebook đều báo sẵn sàng, StoryForge tự chọn đúng máy vừa liên kết khi không có lượt sinh/Auto Write. Bấm **Thử kết nối** để kiểm tra Direct; **Dùng máy này** cho phép thử chọn lại khi cần.
5. Chạy lại cùng ô sẽ tiếp tục phiên đang chờ hoặc đang hoạt động; không tạo pairing khác.

Notebook không yêu cầu Colab Secrets và không nhúng StoryForge session token, Cloudflare API key hoặc credential mẫu.
Sau khi xác nhận, StoryForge cấp API key AI, control secret và connector token một lần. Chúng chỉ ở bộ nhớ runtime;
gateway chỉ nhận API key và đường dẫn tới file lease không chứa secret. Không chia sẻ runtime hoặc output của notebook.

StoryForge cấp hostname riêng và remote tunnel có ingress loopback tới gateway `127.0.0.1:18081`.
Notebook không dùng hostname cá nhân cố định và không mở cổng model `18080` ra Internet. Thu hồi hoặc hết lease sẽ đóng gateway/tunnel;
stream đang chạy bị dừng fail-closed. Nếu runtime mất mạng, lease cũ tối đa 120 giây; heartbeat 60 giây; phiên tối đa 8 giờ.

Mặc định model là `HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive` / `Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf`,
Các lựa chọn Q/IQ ghim revision `45b6a334b4bcd1d7f37179df58b3b1d66a184e5d`, thiết bị **Tự động**, context **128000 mỗi lượt**, max output **64000**, **3 slot**. GGUF khác vẫn dùng ba trường nhập riêng. Chọn trong form chưa tải model; chỉ tải/nạp khi chủ động chạy. Đây là giá trị cấu hình, không phải benchmark.
Chưa có bằng chứng inference thực tế đạt đủ 128000/64000 trên Colab; khả năng chạy tùy RAM/VRAM và thời lượng phiên.

Để tạo lại mã sau khi thu hồi: thu hồi máy cũ trong StoryForge, đợi lease hết hạn (tối đa 120 giây), rồi chọn **Tạo mã liên kết mới**.
Notebook không tự nhận lại credential nếu claim response bị mất; khi đó phải tạo pairing mới. Không tự retry request inference.


In [ ]:
#@title Máy AI cá nhân StoryForge · kết nối bằng mã xác nhận
#@markdown Chọn **Ghép nối / khởi động**, bấm **▶ Chạy**, rồi nhập mã hiện ra vào StoryForge và xác nhận.
thao_tac = "Ghép nối / khởi động" #@param ["Ghép nối / khởi động", "Khôi phục cấu hình trước", "Tạo mã liên kết mới", "Xem trạng thái", "Chẩn đoán", "Dừng máy chủ"]
#@markdown ### Model và thiết bị
model_AI = "Q8_K_P" #@param ["IQ3_M", "IQ4_XS", "Q2_K_P", "Q3_K_M", "Q3_K_P", "Q4_K_M", "Q4_K_P", "Q5_K_M", "Q5_K_P", "Q6_K_P", "Q8_K_P", "GGUF khác"]
thiet_bi = "Tự động" #@param ["Tự động", "GPU", "CPU"]
so_yeu_cau_dong_thoi = 3 #@param [1, 3, 5] {type:"raw"}
#@markdown **3 lượt đồng thời** mặc định; **5 là thử nghiệm**, **1** để quay về tải thấp. Hàng đợi tối đa 5, chờ tối đa 60 giây rồi từ chối. Tăng slot sẽ nạp lại model và tăng RAM/VRAM; không tự giảm cấu hình khi thiếu bộ nhớ.
#@markdown ### Ngữ cảnh và độ dài trả lời
#@markdown **128000 context mỗi lượt / 64000 đầu ra tối đa mỗi lượt**. Pool context = context × số slot (3: 384000; 5: 640000). Context gồm đầu vào, lịch sử, hướng dẫn và đầu ra; không bắt mỗi yêu cầu sinh đủ giới hạn. Chưa benchmark đủ độ dài hoặc bộ nhớ trên GPU của bạn.
context_tokens = 128000 #@param {type:"integer"}
max_output_tokens = 64000 #@param {type:"integer"}
#@markdown ### GGUF
hf_repo = "HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive" #@param {type:"string"}
gguf_file = "Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf" #@param {type:"string"}
hf_revision = "main" #@param {type:"string"}
#@markdown Không cần Colab Secrets. Notebook chỉ nhận credential sau khi người dùng xác nhận mã trong StoryForge.
# File này do build_multiuser_notebook.py sinh; origin không thể đổi trong biểu mẫu.
import base64, io, zipfile, sys, importlib, importlib.metadata, subprocess
SF_ORIGIN = 'https://storyforge-ai-pilot.canhettg113.workers.dev'
if SF_ORIGIN.endswith('.invalid'):
    raise RuntimeError('REVIEW ONLY: origin mẫu .invalid chưa kết nối được tới StoryForge; không gọi mạng.')
try:
    _sf_httpx_version = importlib.metadata.version('httpx')
except importlib.metadata.PackageNotFoundError:
    _sf_httpx_version = None
if _sf_httpx_version != '0.28.1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'httpx==0.28.1'],
                   check=True, timeout=180)
from pathlib import Path
_sf_folder = Path('/content/storyforge-personal-ai/source')
_sf_folder.mkdir(parents=True, exist_ok=True)
_sf_payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_sf_payload))) as _sf_zip:
    _sf_zip.extractall(_sf_folder)
if str(_sf_folder) not in sys.path:
    sys.path.insert(0, str(_sf_folder))
import runtime, connection, settings, linked_connection, start_colab
for _sf_module in (runtime, connection, settings, linked_connection, start_colab):
    importlib.reload(_sf_module)
_sf_personal_state = globals().get('SF_STATE', {})
if isinstance(_sf_personal_state, dict) and any(
        _sf_personal_state.get(_sf_name) is not None
        and _sf_personal_state[_sf_name].poll() is None
        for _sf_name in ('model', 'gateway', 'tunnel')):
    raise RuntimeError('Dừng máy chủ notebook cá nhân trong runtime này trước khi ghép nối nhiều người dùng.')
start_colab.ROOT = Path('/content/storyforge-personal-ai')
SF_LINKED_STATE = globals().get('SF_LINKED_STATE', {})
SF_LINKED_STATE['multiuser'] = True
_sf_settings = {'model_choice': model_AI, 'device': thiet_bi,
    'parallel': so_yeu_cau_dong_thoi,
    'context': context_tokens, 'max_tokens': max_output_tokens,
    'repo': hf_repo, 'filename': gguf_file, 'revision': hf_revision}

def _sf_wait_message(status):
    labels = {'pending': 'Đang chờ Anh/Chị xác nhận trong StoryForge…',
              'approved': 'Đã xác nhận. StoryForge đang chuẩn bị kết nối…',
              'provisioning': 'Đang cấp đường kết nối riêng…',
              'slow_down': 'Máy chủ yêu cầu chờ lâu hơn trước lần kiểm tra tiếp theo…'}
    if status in labels:
        print(labels[status], flush=True)

try:
    if thao_tac == 'Xem trạng thái':
        start_colab.show_linked_panel(SF_LINKED_STATE)
    elif thao_tac == 'Chẩn đoán':
        start_colab.diagnostics(SF_LINKED_STATE)
    elif thao_tac == 'Dừng máy chủ':
        start_colab.stop(SF_LINKED_STATE)
        print('Đã dừng các tiến trình do notebook quản lý. Kết nối sẽ hết lease nếu không còn heartbeat.')
    else:
        if thao_tac == 'Khôi phục cấu hình trước':
            _sf_previous = SF_LINKED_STATE.get('previous_model_config')
            if not _sf_previous:
                raise ValueError('Chưa có cấu hình trước để khôi phục.')
            _sf_settings = {'model_choice': 'GGUF khác', 'device': 'GPU' if _sf_previous['device'] == 'gpu' else 'CPU',
                'parallel': _sf_previous.get('parallel', 1),
                **{key: _sf_previous[key] for key in ('context', 'max_tokens', 'repo', 'filename', 'revision')}}
        _sf_link = linked_connection.LinkedConnection(SF_ORIGIN, SF_LINKED_STATE)
        if thao_tac == 'Tạo mã liên kết mới':
            start_colab.prepare_new_link(SF_LINKED_STATE)
            _sf_pair = _sf_link.new_pair(label='Colab AI')
        else:
            _sf_pair = _sf_link.ensure_pair(label='Colab AI')
        if _sf_pair:
            print('Mã liên kết dùng một lần:', _sf_pair['code'], flush=True)
            print('Nhập mã này trong Cài đặt kết nối máy AI của StoryForge và xác nhận rõ ràng.', flush=True)
        if SF_LINKED_STATE.get('linked_credentials'):
            _sf_credentials = SF_LINKED_STATE['linked_credentials']
        else:
            print('Đang chờ xác nhận; lần chạy lại sẽ tiếp tục cùng mã liên kết.', flush=True)
            _sf_credentials = _sf_link.poll_until_claim(max_wait=600, on_status=_sf_wait_message)
        print('Đã nhận quyền phiên trong bộ nhớ. Credential không được in ra notebook.', flush=True)
        start_colab.start_linked(SF_LINKED_STATE, _sf_credentials, SF_ORIGIN, settings=_sf_settings)
        start_colab.show_linked_panel(SF_LINKED_STATE)
except Exception as _sf_error:
    if ((isinstance(_sf_error, (ValueError, TimeoutError)) and type(_sf_error).__name__ != 'JSONDecodeError')
            or isinstance(_sf_error, runtime.RuntimeActionError)):
        print('Chưa áp dụng yêu cầu:', str(_sf_error))
    else:
        print('CHƯA SẴN SÀNG:', type(_sf_error).__name__)
        print('Chi tiết thô và nhật ký riêng tư được ẩn. Chọn Chẩn đoán để xem trạng thái tiến trình.')
    start_colab.show_linked_panel(SF_LINKED_STATE)
